In [ ]:
%pip install --upgrade --quiet google-genai

In [ ]:
%pip install pypdf

In [ ]:
import os
import sys

from IPython.display import HTML, Markdown, display
from google import genai
from google.genai import types
from google.genai.types import GenerateContentConfig, Part
from pydantic import BaseModel

import json
import pypdf
from pypdf import PdfReader, PdfWriter
import time
import pathlib
from pathlib import Path
import re

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

In [ ]:
# fmt: off
PROJECT_ID = "project-31bcebfe-0bcc-44c6-9c2"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = "global"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [ ]:
# List all available Gemini models
for model in client.models.list():
    print(f"Model: {model.name}")
    print("---")

Model: publishers/google/models/gemini-1.5-pro-002
---
Model: publishers/google/models/gemini-2.0-flash-001
---
Model: publishers/google/models/gemini-2.0-flash
---
Model: publishers/google/models/gemini-2.0-flash-lite-001
---
Model: publishers/google/models/gemini-2.5-flash-preview-04-17
---
Model: publishers/google/models/gemini-2.5-pro-exp-03-25
---
Model: publishers/google/models/gemini-2.5-pro
---
Model: publishers/google/models/gemini-2.5-flash
---
Model: publishers/google/models/gemini-2.5-flash-lite
---
Model: publishers/google/models/gemini-2.5-pro-tts
---
Model: publishers/google/models/gemini-2.5-flash-tts
---
Model: publishers/google/models/gemini-live-2.5-flash-native-audio
---
Model: publishers/google/models/gemini-3-flash-preview
---
Model: publishers/google/models/gemini-3.1-flash-lite-preview
---
Model: publishers/google/models/gemini-3.1-flash-image-preview
---
Model: publishers/google/models/gemini-3.1-pro-preview
---
Model: publishers/google/models/gemini-embedding-

In [ ]:
MODEL_ID = "gemini-3-flash-preview"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# go to specific location
path = '/content/drive/MyDrive/merve-february2026'
os.chdir(path)

# Confirm the change
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/merve-february2026


In [ ]:
# Split the PDF into pages

def extract_pdf_pages(pdf_path, output_dir, starting_page_num=1):
    """
    Extract all pages from a PDF and save them as individual PDF files.

    Parameters:
    -----------
    pdf_path : str
        Path to the input PDF file
    starting_page_num : int, default=1
        The starting page number for naming the output files

    Returns:
    --------
    str
        Path to the output folder containing the individual page PDFs

    Example:
    --------
    >>> extract_pdf_pages("document.pdf", starting_page_num=5)
    # This will create files: page_5.pdf, page_6.pdf, page_7.pdf, etc.
    """

    # Hardcoded output folder name
    output_folder = "transliteration_pages"

    output_path = output_dir + "/" + output_folder

    # Create output folder if it doesn't exist
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    # Read the PDF
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)

    print(f"Processing {total_pages} pages from '{pdf_path}'...")

    # Extract each page
    for page_index in range(total_pages):
        # Create a new PDF writer for each page
        writer = PdfWriter()

        # Add the current page to the writer
        writer.add_page(reader.pages[page_index])

        # Calculate the output page number
        output_page_num = starting_page_num + page_index

        # Create output filename
        output_filename = f"page_{output_page_num}.pdf"
        output_page_path = os.path.join(output_path, output_filename)

        # Write the page to a new PDF file
        with open(output_page_path, 'wb') as output_file:
            writer.write(output_file)

    print(f"\nDone! All {total_pages} pages saved to '{output_path}/' folder")
    return output_folder

In [ ]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/abdi_1729-1731.pdf', 'abdi_1729-1731', 3)

Processing 65 pages from '/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/abdi_1729-1731.pdf'...

Done! All 65 pages saved to 'abdi_1729-1731/transliteration_pages/' folder


'transliteration_pages'

In [ ]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1730-1755.pdf', 'semdanizadesuleyman_1730-1755', 1)

Processing 182 pages from '/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1730-1755.pdf'...

Done! All 182 pages saved to 'semdanizadesuleyman_1730-1755/transliteration_pages/' folder


'transliteration_pages'

In [ ]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1755-1769.pdf', 'semdanizadesuleyman_1755-1769', 3)

Processing 124 pages from '/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1755-1769.pdf'...

Done! All 124 pages saved to 'semdanizadesuleyman_1755-1769/transliteration_pages/' folder


'transliteration_pages'

In [ ]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1769-1774.pdf', 'semdanizadesuleyman_1769-1774', 3)

Processing 118 pages from '/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1769-1774.pdf'...

Done! All 118 pages saved to 'semdanizadesuleyman_1769-1774/transliteration_pages/' folder


'transliteration_pages'

In [ ]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1774-1777.pdf', 'semdanizadesuleyman_1774-1777', 3)

Processing 65 pages from '/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/semdanizadesuleyman_1774-1777.pdf'...

Done! All 65 pages saved to 'semdanizadesuleyman_1774-1777/transliteration_pages/' folder


'transliteration_pages'

In [65]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/maintext_montagu_fr_1764.pdf', 'montagu_fr_1764', 9)

Processing 513 pages from '/content/drive/MyDrive/merve-february2026/maintext_montagu_fr_1764.pdf'...

Done! All 513 pages saved to 'montagu_fr_1764/transliteration_pages/' folder


'transliteration_pages'

In [66]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/maintext_montagu_1816_fr-en_v1.pdf', 'montagu_1816_fr-en_v1', 2)

Processing 404 pages from '/content/drive/MyDrive/merve-february2026/maintext_montagu_1816_fr-en_v1.pdf'...

Done! All 404 pages saved to 'montagu_1816_fr-en_v1/transliteration_pages/' folder


'transliteration_pages'

In [67]:
extract_pdf_pages('/content/drive/MyDrive/merve-february2026/maintext_montagu_1816_fr-en_v2.pdf', 'montagu_1816_fr-en_v2', 2)

Processing 354 pages from '/content/drive/MyDrive/merve-february2026/maintext_montagu_1816_fr-en_v2.pdf'...

Done! All 354 pages saved to 'montagu_1816_fr-en_v2/transliteration_pages/' folder


'transliteration_pages'

In [68]:
# Directory containing the examples
#examples_dir = 'few_shot_examples'
#examples_dir = 'kirimi_fewshot'
#examples_dir = 'table_fewshot'
#examples_dir = 'marginalia_fewshot'
examples_dir = 'montagu_french_fewshot'

# Get all PDF files in the directory
pdf_files = [f for f in os.listdir(examples_dir) if f.endswith('.pdf')]

# Extract page numbers from filenames
page_numbers = []
for pdf_file in pdf_files:
    # Extract number from filename like "page_35.pdf"
    page_num = int(pdf_file.replace('page_', '').replace('.pdf', ''))
    page_numbers.append(page_num)

# Sort page numbers for consistent ordering
page_numbers.sort()

# Dictionary to store PDF data as Parts
example_pdfs = {}

# Load each PDF as bytes and create Parts
for page_num in page_numbers:
    pdf_path = os.path.join(examples_dir, f'page_{page_num}.pdf')

    with open(pdf_path, 'rb') as f:
        file_bytes = f.read()

    # Store as Part objects
    example_pdfs[page_num] = Part.from_bytes(
        data=file_bytes,
        mime_type="application/pdf"
    )

# Load the corresponding JSON files and combine with PDFs
example_parts = []
for page_num in page_numbers:
    json_path = os.path.join(examples_dir, f'page_{page_num}.json')

    with open(json_path, 'r', encoding='utf-8') as f:
        output_data = json.load(f)

    output_json = json.dumps(output_data, ensure_ascii=False, indent=2)

    example_parts.append({
        "file": example_pdfs[page_num],
        "page": page_num,
        "output_json": output_json
    })

print(f"Loaded {len(example_parts)} examples from {examples_dir}/")
print(f"Pages: {page_numbers}")

Loaded 6 examples from montagu_french_fewshot/
Pages: [16, 17, 22, 24, 29, 31]


In [ ]:
instructions = """You are an expert of Ottoman Turkish.
Your task is to extract texts from the transliterated manuscript that is provided as images.
    - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
    - Refer to the examples provided to understand the expected JSON structure.

<INSTRUCTIONS>
- Identify all events on the page.
- Each event has a subheading (standalone bold or centered text) and body text.
- A single event can cover multiple paragraphs. A new paragraph does NOT signal a new event. If an event contains internal divisions like numbered articles (e.g., “İkinci mâdde:”, “Üçüncü mâdde:”), preserve these as paragraph breaks (`\n\n`) within a single `body` field.
- Some events may cover multiple pages. If text continues from a previous page, set subheading to null.
- Treat short formulaic labels (e.g., “Birinci Madde:“, “Amma ba‘dü”, “Nazm:”, “[Mısra]”) as inline markers within the body even if they are bold, unless they appear as a standalone heading line.
- If a subheading is followed by a footnote marker that provides metadata (e.g., manuscript variants like marginalia notes), extract that into a `subheading_notes` field.
- **Text Normalization**:
    - Join words split by a hyphen at the end of a line (e.g., "müste-sna" becomes "müstesna") into a single word in the body text.
    - If the manuscript uses older typographic conventions (like a superscript 'c' for ʿayn), normalize it to 'ʿ'.
- Preserve manuscript page numbers like [12a], (18b), or [26] in the body text exactly where they appear.
- Some manuscript page numbers are only marked with bold script. If that is the case put bold page markers into brackets (e.g., "*7b*" becomes [7b])
- Exclude footnotes (numbered annotations at bottom of page) and remove footnote markers (superscript numbers) from the body text.
- Year headings (all-caps, bold) should be events with only a subheading and null body.
</INSTRUCTIONS>

You will see examples below, then the page to process."""

In [69]:
# for montagu

instructions = """You are a specialist in 18th and 19th-century French and English typography.
Transcribe the scanned page exactly as instructed and return ONLY valid JSON — no markdown fences, no explanation.

STRUCTURE
---------
Every page returns a JSON object with these top-level fields:

  page          : Page number exactly as printed. Integer for arabic numerals (24, 31),
                  string for roman numerals ("viij", "ix"). Do not infer — read it from the page.
  language      : The language of the letter, either English marked as "en" or French marked as "fr"
  letters       : Always a list. Never omit this field. Rules below.

LETTERS LIST
------------
Each item in "letters" represents one letter's content visible on this page.

  number    : The letter number as printed (e.g. "V", "VI", "IV"). null if this item is a
              continuation fragment with no letter opening on this page.
  heading   : The heading line as printed (e.g. "LETTRE V.", "IV. TO THE LADY RICH.*",
              "IV. A LADY RICH*."). null if number is null.
  recipient : The recipient line as printed (e.g. "A la Comtesse de B***.", "A Madame P***.").
              null if number is null. For editions where the recipient is embedded in the heading
              line rather than printed separately, set to null.
  location  : The place of writing as printed (e.g. "Nuremberg", "Cologn", "Cologne"). null if
              number is null.
  date      : The date line as printed (e.g. "le 22 Août 1716. Vieux style.",
              "Aug. 16, O. S. 1716.", "le 16 août 1716, V. S."). null if number is null.
  body      : All body text of this letter visible on this page. Never null — use "" only if
              genuinely empty. If the body ends mid-word due to a page break, include the
              hyphenated fragment (e.g. "...femme de l'En-", "...that no enchant-").
  footnote  : A single string containing the footnote(s) for this letter on this page. If there
              are multiple footnotes for one letter on one page, concatenate them separated by
              " / ". null if there is no footnote.

HOW MANY ITEMS IN LETTERS
--------------------------
- Pure continuation page (no letter opens, no letter closes): one item, all metadata null.
- Page where a single letter opens: one item, with number/heading/recipient/location/date filled.
- Page where one letter ends and another begins: two items. The first item has all metadata null
  (it is the tail of the previous letter). The second item has full metadata for the new letter.
- Section/front-matter page: one item with all metadata null, body contains the prose text.
- Blank page: empty list [].

RUNNING HEADERS
---------------
Running headers (e.g. "LETTRE IV." or "LETTER XLII." printed in the top margin) are NOT
letter headings. Ignore them entirely — do not put them in heading and do not let them
influence number or any other field.

QUIRE SIGNATURES
----------------
Short letter+numeral marks printed below and to the right of the text block (e.g. "B iij",
"B iv", "A iv", "C", "D iv") are printer's gathering marks. Ignore them entirely — they are
not part of the body text and must not appear anywhere in the output.

IMAGES
----------------
There are occasional etchings and other forms of adornment. Ignore them entirely.

ORTHOGRAPHY
-----------
- Normalize long-s (ſ) to regular s.
- Preserve & (ampersand), all accents, and period punctuation faithfully.
- Do not modernize spelling in either language.
- For the bilingual 1816 edition: transcribe each page exactly as printed in its own language
  with no normalization.

FOOTNOTES
---------
- Footnote markers (* † ‡ or numbers) must be preserved inline in body exactly where they appear.
- The full footnote text goes in the footnote field of the letter item they annotate.
- If a footnote belongs to a continuation fragment (number: null), place it in that item's
  footnote field.

NULL VS ABSENT
--------------
All fields listed above must always be present. Use null explicitly — never omit a field.
footnote is null (not "") when absent. body is never null — use "" only for genuinely empty pages."""

In [ ]:
# for marginalia

instructions = """You are an expert of Ottoman Turkish.
Your task is to extract texts from the transliterated manuscript that is provided as images.
    - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
    - Refer to the examples provided to understand the expected JSON structure.

<INSTRUCTIONS>
- Identify all events on the page.
- Each event has a subheading (standalone bold or centered text) and body text.
- A single event can cover multiple paragraphs. A new paragraph does NOT signal a new event. If an event contains internal divisions like numbered articles (e.g., "İkinci mâdde:", "Üçüncü mâdde:"), preserve these as paragraph breaks (`\n\n`) within a single `body` field.
- Some events may cover multiple pages. If text continues from a previous page, set subheading to null.
- Treat short formulaic labels (e.g., "Birinci Madde:", "Amma ba'dü", "Nazm:", "[Mısra]") as inline markers within the body even if they are bold, unless they appear as a standalone heading line.
- If a subheading is followed by a footnote marker that provides metadata (e.g., manuscript variants like marginalia notes), extract that into a `subheading_notes` field.
- **Marginalia**: Text printed in the margin — whether in small-caps, bold, or italic — are section labels, not subheadings. They annotate the adjacent body text but are not part of the narrative flow. Collect all marginalia found on the page into a top-level `marginalia` array. Do NOT include them in the body text. A label is marginalia if it: (a) appears physically in the left or right margin rather than inline with the body text, (b) summarizes or titles the adjacent passage rather than continuing the narrative, or (c) is typographically distinct (small-caps, italic, or bold) and set apart from the main text column. A label is a subheading if it appears as a standalone centered or indented line within the main text column and introduces a new named section.
- **Year headings**: Year labels (e.g., "Sene: 1170 (1756-1757)") should be treated as subheadings. The annalistic entry text that follows — typically a list of events recorded under that year — should be captured as the `body`. Preserve the year label exactly as it appears in the manuscript.
- **Text Normalization**:
    - Join words split by a hyphen at the end of a line (e.g., "müste-sna" becomes "müstesna") into a single word in the body text.
    - If the manuscript uses older typographic conventions (like a superscript 'c' for ʿayn), normalize it to 'ʿ'.
- Preserve manuscript page numbers like [12a], (18b), or [26] in the body text exactly where they appear.
- Some manuscript page numbers are only marked with bold script. If that is the case put bold page markers into brackets (e.g., "*7b*" becomes [7b])
- Exclude footnotes (numbered annotations at bottom of page) and remove footnote markers (superscript numbers) from the body text.
</INSTRUCTIONS>

You will see examples below, then the page to process."""

In [ ]:
# for tables in layiha to be used with table few shot

instructions = """You are an expert of Ottoman Turkish.
Your task is to extract tables from transliterated Ottoman manuscript pages provided as images and convert them into structured JSON.
    - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
    - Refer to the examples provided to understand the expected JSON structure.

<INSTRUCTIONS>
- Identify all tables on the page.
- Each table entry has a `page` and a `table` field.
- A single page may contain more than one table. If so, produce one JSON object per table, each with the correct page number.
- The `page` field should reflect the manuscript folio number as it appears on the page (e.g., "99/b", "3", "171/b").
- The `table` field contains the full table as a markdown string, with rows separated by `\n`.
- **Context rows**: Any descriptive or explanatory text that appears inside the table borders — whether as a header, introductory paragraph, or narrative continuation — must be included as a full-width row in the markdown table. Do not discard it.
- **Column headers**: Reproduce column headers exactly as they appear, including rotated or stacked headers. If headers span multiple levels, represent each level as its own row.
- **Empty cells**: Represent empty cells with an empty markdown cell (i.e., `| |`). Do not skip columns.
- **Numeric values**: Preserve leading zeros and spacing exactly as they appear (e.g., `02019`, `0090`).
- **Ottoman Turkish**: Preserve all diacritics, special characters, and transliteration conventions exactly (e.g., â, î, û, ʿ, ğ).
- **Text Normalization**: Do not normalize or modernize spelling. Transcribe exactly as printed.
- Exclude any text that appears outside the table borders (e.g., page headers, editorial stamps, or marginal notes).
</INSTRUCTIONS>

You will see examples below, then the page to process."""

In [ ]:
# instructions = """You are an expert of Ottoman Turkish.
# Your task is to extract texts from the transliterated manuscript that is provided as images.
#     - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
#     - Refer to the examples provided to understand the expected JSON structure.

# <INSTRUCTIONS>
# - Identify all events on the page.
# - Each event has a subheading (standalone bold or centered text) and body text.
# - A single event can cover multiple paragraphs. A new paragraph does NOT signal a new event. If an event contains internal divisions like numbered articles (e.g., “İkinci mâdde:”, “Üçüncü mâdde:”), preserve these as paragraph breaks (`\n\n`) within a single `body` field.
# - Some events may cover multiple pages. If text continues from a previous page, set subheading to null.
# - Treat short formulaic labels (e.g., “Birinci Madde:“, “Amma ba‘dü”, “Nazm:”, “[Mısra]”) as inline markers within the body even if they are bold, unless they appear as a standalone heading line.
# - If a subheading is followed by a footnote marker that provides metadata (e.g., manuscript variants like marginalia notes), extract that into a `subheading_notes` field.
# - **Text Normalization**:
#     - Join words split by a hyphen at the end of a line (e.g., "müste-sna" becomes "müstesna") into a single word in the body text.
#     - If the manuscript uses older typographic conventions (like a superscript 'c' for ʿayn), normalize it to 'ʿ'.
# - Preserve manuscript page numbers like [12a], (18b), or [26] in the body text exactly where they appear.
# - Some manuscript page numbers are only marked with bold script. If that is the case put bold page markers into brackets (e.g., "*7b*" becomes [7b])
# - Exclude footnotes (numbered annotations at bottom of page) and remove footnote markers (superscript numbers) from the body text.
# - Year headings (all-caps, bold) should be events with only a subheading and null body.
# - if there are any images, ignore the images and their captions.
# </INSTRUCTIONS>

# You will see examples below, then the page to process."""

In [ ]:
# # for ebubekir layiha to be used with exampes 16, 29, 280

# instructions = """You are an expert of Ottoman Turkish.
# Your task is to extract texts from the transliterated manuscript that is provided as images.
#     - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
#     - Refer to the examples provided to understand the expected JSON structure.

# <INSTRUCTIONS>
# - Identify all events on the page.
# - Each event has a subheading (standalone bold or centered text) and body text.
# - A single event can cover multiple paragraphs. A new paragraph does NOT signal a new event. If an event contains internal divisions like numbered articles (e.g., “İkinci mâdde:”, “Üçüncü mâdde:”), preserve these as paragraph breaks (`\n\n`) within a single `body` field.
# - Some events may cover multiple pages. If text continues from a previous page, set subheading to null.
# - Treat short formulaic labels (e.g., “Birinci Madde:“, “Amma ba‘dü”, “Nazm:”, “[Mısra]”) as inline markers within the body even if they are bold, unless they appear as a standalone heading line.
# - If a subheading is followed by a footnote marker that provides metadata (e.g., manuscript variants like marginalia notes), extract that into a `subheading_notes` field.
# - **Text Normalization**:
#     - Join words split by a hyphen at the end of a line (e.g., "müste-sna" becomes "müstesna") into a single word in the body text.
#     - If the manuscript uses older typographic conventions (like a superscript 'c' for ʿayn), normalize it to 'ʿ'.
# - Preserve manuscript page numbers like [12a], (18b), or [26] in the body text exactly where they appear.
# - Some manuscript page numbers are only marked with bold script. If that is the case put bold page markers into brackets (e.g., "*7b*" becomes [7b])
# - Exclude footnotes (numbered annotations at bottom of page) and remove footnote markers (superscript numbers) from the body text.
# - There are some handwritten corrections. Accept minor corrections and edits like "ve'l-rızâ" in text to "ve'r-rızâ" in post-print edits. Ignore all long comments and questions in modern Turkish.
# - Year headings (all-caps, bold) should be events with only a subheading and null body.
# </INSTRUCTIONS>

# You will see examples below, then the page to process."""

In [ ]:
# instructions for the new few-shot examples where there is a subheading in the margins like taylesanizadeabdullah_1785-1789
# use with few-shot examples 65, 74

instructions = """You are an expert of Ottoman Turkish.
Your task is to extract texts from the transliterated manuscript that is provided as images.
    - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
    - Refer to the examples provided to understand the expected JSON structure.

<INSTRUCTIONS>
- Identify all events on the page.
- **Subheading Identification**:
    - Subheadings may appear as standalone bold/centered lines **OR** as italicized text in the left margin next to a paragraph.
    - If a subheading is in the margin, map it to the `subheading` field and the text immediately to its right to the `body` field.
- **Chronological Markers**:
    - Centered headings for a specific month or year (e.g., "Vekāyi‘-i şehr-i...") must be treated as separate events.
    - For these markers, set the full heading as the `subheading` and set the `body` to `null`.
- **Event Continuity**:
    - A single event can cover multiple paragraphs. A new paragraph does NOT signal a new event unless a new subheading is present.
    - If text continues from a previous page without a new heading, set `subheading` to `null`.
    - Internal divisions like numbered articles (e.g., “İkinci mâdde:”) should be preserved as paragraph breaks (`\n\n`) within the `body`.
- **Text Normalization & Paleography**:
    - Use the inverted comma ` ‘ ` for ‘Ayn (ع) and the curly apostrophe ` ’ ` for Hemze (ء) (e.g., "mi’ete").
    - Join words split by a hyphen at the end of a line (e.g., "müste-sna" becomes "müstesna").
    - Preserve manuscript page numbers (e.g., [12a], [26]) exactly where they appear in the text.
- **Exclusions**:
    - Exclude footnotes at the bottom of the page and remove superscript footnote markers from the body text.
    - If a subheading has a metadata marker (manuscript variants), extract it into a `subheading_notes` field.
</INSTRUCTIONS>

You will see examples below, then the page to process."""

In [ ]:
# # instructions only for destarisalih_vakaiibretnuma, make sure to add page_1 set to the few shot

# instructions = """You are an expert of Ottoman Turkish.
# Your task is to extract texts from the transliterated manuscript that is provided as images.
#     - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
#     - Refer to the examples provided to understand the expected JSON structure.

# <INSTRUCTIONS>
# - Identify all events on the page.
# - Each event has a subheading (standalone bold or centered text) and body text.
# - A single event can cover multiple paragraphs. A new paragraph does NOT signal a new event. If an event contains internal divisions like numbered articles (e.g., “İkinci mâdde:”, “Üçüncü mâdde:”), preserve these as paragraph breaks (`\n\n`) within a single `body` field.
# - Some events may cover multiple pages. If text continues from a previous page, set subheading to null.
# - Treat short formulaic labels (e.g., “Birinci Madde:“, “Amma ba‘dü”, “Nazm:”, “[Mısra]”) as inline markers within the body even if they are bold, unless they appear as a standalone heading line.
# - If a subheading is followed by a footnote marker that provides metadata (e.g., manuscript variants like marginalia notes), extract that into a `subheading_notes` field.
# - **Text Normalization**:
#     - Normalize editorial insertions like "<ve>" to square brackets "[ve]".
#     - Normalize the Persian İzafet marker (represented as a low curve or underscore "_") to a standard hyphen "-".
#     - Join words split by a hyphen at the end of a line (e.g., "müste-sna" becomes "müstesna") into a single word in the body text.
#     - If the manuscript uses older typographic conventions (like a superscript 'c' for ʿayn), normalize it to 'ʿ'.
# - Preserve manuscript page numbers like [12a], (18b), or [26] in the body text exactly where they appear.
# - Exclude footnotes (numbered annotations at bottom of page) and remove footnote markers (superscript numbers) from the body text.
# - Year headings (all-caps, bold) should be events with only a subheading and null body.
# </INSTRUCTIONS>

# You will see examples below, then the page to process."""

In [ ]:
# # instructions for abdulgaffarkirimi_1747, use the kirimi_fewshot folder with this

# instructions = """You are an expert of Ottoman Turkish paleography and transcription.
# Your task is to extract texts from the transliterated, line-numbered manuscript that is provided as images.
#     - You will see one page of the manuscript at a time, but know that it is one continuous manuscript.
#     - Provide a single JSON object for the entire image (PDF page).
#     - Refer to the examples provided to understand the expected JSON structure.

# <INSTRUCTIONS>
# - **Outer Structure**:
#     - Set the "page" field to the numerical PDF page number (e.g., 155, 266).
#     - Provide an "events" field which is a list containing a single object for the entire page.
# - **Event Mapping**:
#     - **subheading**: Enter the manuscript page marker if present (e.g., "s.266-a"). If the page is a continuation without a new marker, set this to null.
#     - **body**:
#         - Transcribe all numbered lines from the image into this single field.
#         - Preserve the line numbers (e.g., "16.", "17.") at the start of their respective lines.
#         - Use exactly two newlines (`\n\n`) between each numbered line to maintain clear separation.
# - **Catch-words:
#     - If a catch-word appears at the bottom right (e.g., "_idi_"), include it as its own event. Set the subheading of the event to the catch-word and leave the body null.
# - **Text Normalization**:
#     - **ʿAyn (ʿ)**: Use the left half-ring `ʿ` for the letter ʿayn (e.g., *baʿde*, *iğtinâm*, *iʿtiqâd*).
#     - **Hamza (ʾ)**: Use the right half-ring `ʾ` for the letter hamza (e.g., *meʾmûr*, *teʾsîr*, *ümerâʾ*).
#     - **Apostrophe (')**: Reserve the standard apostrophe `'` for Turkish possessive suffixes (e.g., *Giray'ın*, *Giray'ı*) and Arabic phonetic links/elisions (e.g., celîlü'ş-şân, sabık'üz-zikr).
#     - **Spaces**: Only add a space if a punctuation mark is masking a clear word boundary (e.g., `Han'a'aks` -> `Han'a ʿaks`).
#     - **İzafet**: Continue using the standard hyphen `-` for Persian links (e.g., *han-ı celîlü'ş-şân*).
# - **Deletions**: Use `~~text~~` for struck-through words.
# </INSTRUCTIONS>

# You will see examples below, then the page to process."""



In [70]:
# second version

def process_page(page_num, client, model_id, instructions, example_parts, base_dir):
    """
    Process a single page and return both extracted JSON and usage metadata.

    Args:
        page_num: Page number to process
        client: Gemini client
        model_id: Model identifier
        instructions: Processing instructions
        example_parts: Few-shot examples
        base_dir: Base directory containing the transliteration_pages folder

    Returns: (success: bool, data: dict)
    """

    pdf_path = Path(base_dir) / 'transliteration_pages' / f'page_{page_num}.pdf'

    if not pdf_path.exists():
        return False, {"page": page_num, "error": f"PDF not found: {pdf_path}"}

    # Read the PDF as bytes
    try:
        with open(pdf_path, "rb") as f:
            page_bytes = f.read()
    except Exception as e:
        return False, {"page": page_num, "error": f"Failed to read PDF: {str(e)}"}

    page_part = Part.from_bytes(data=page_bytes, mime_type="application/pdf")

    # Build the prompt using the few-shot examples
    contents = [instructions, "\n<EXAMPLES>\n"]
    for part in example_parts:
        contents.extend([
            part["file"],
            f"Output for page {part['page']}:",
            part["output_json"],
        ])
    contents.extend(["\n</EXAMPLES>\n", "\n--- PROCESS THIS PAGE ---\n"])
    contents.append(page_part)

    # Call Gemini
    try:
        response = client.models.generate_content(
            model=model_id,
            contents=contents,
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="low"),
                safety_settings=[
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                ],
            ),
        )

        # Extract response text safely
        try:
            raw_text = response.text.strip() if response.text else ""
        except (ValueError, AttributeError):
            raw_text = ""

        # Get finish reason
        finish_reason = (
            response.candidates[0].finish_reason.name
            if response.candidates
            else "NO_CANDIDATES"
        )

        # Determine success based on whether we got text
        is_successful = bool(raw_text)

        result = {
            "page": page_num,
            "raw_response": raw_text,
            "finish_reason": finish_reason,
            "usage": {
                "prompt_tokens": response.usage_metadata.prompt_token_count,
                "thought_tokens": response.usage_metadata.thoughts_token_count,
                "output_tokens": response.usage_metadata.candidates_token_count,
                "total_tokens": response.usage_metadata.total_token_count,
            }
        }

        # Add safety info if response was blocked/empty
        if not is_successful:
            try:
                # Try to get safety ratings from candidates first
                if hasattr(response, 'candidates') and response.candidates:
                    result["safety_ratings"] = [
                        {"category": rating.category.name, "probability": rating.probability.name}
                        for rating in response.candidates[0].safety_ratings
                    ]

                # Also check prompt_feedback for blocking info (often contains safety data when NO_CANDIDATES)
                if hasattr(response, 'prompt_feedback') and response.prompt_feedback:
                    prompt_feedback = {}

                    if hasattr(response.prompt_feedback, 'block_reason'):
                        prompt_feedback["block_reason"] = response.prompt_feedback.block_reason.name

                    if hasattr(response.prompt_feedback, 'safety_ratings'):
                        prompt_feedback["safety_ratings"] = [
                            {"category": rating.category.name, "probability": rating.probability.name}
                            for rating in response.prompt_feedback.safety_ratings
                        ]

                    if prompt_feedback:
                        result["prompt_feedback"] = prompt_feedback

            except Exception as e:
                # Don't fail the whole request if we can't get safety info
                result["safety_info_error"] = str(e)

            # Add error field
            result["error"] = f"Empty response (finish_reason: {finish_reason})"

        return is_successful, result

    except Exception as e:
        return False, {"page": page_num, "error": str(e)}


In [71]:
# second version

def batch_process_pages(start_page, end_page, client, model_id, instructions, example_parts, base_dir, skip_existing=True):
    """
    Process pages with comprehensive retry logic.

    Retries for:
    - Rate limit errors (429/RESOURCE_EXHAUSTED)
    - Empty responses (blocked/safety issues)
    - API errors (NoneType, etc.)
    """
    output_dir = Path(base_dir) / 'gemini_output'
    output_dir.mkdir(parents=True, exist_ok=True)

    jsonl_file = output_dir / 'all_responses.jsonl'
    results_summary = []

    # Retry parameters
    base_delay = 5
    max_delay = 120
    max_retries_rate_limit = 5
    max_retries_other = 3

    for page_num in range(start_page, end_page + 1):
        individual_file = output_dir / f'page_{page_num}.json'

        if skip_existing and individual_file.exists():
            print(f"⊘ Page {page_num} already exists, skipping...")
            results_summary.append({
                "page": page_num,
                "status": "skipped",
                "reason": "already exists"
            })
            continue

        print(f"Processing page {page_num}...")

        # Retry loop
        retry_count = 0
        success = False
        final_data = None
        wait_time = base_delay

        while retry_count < max_retries_rate_limit:
            success, data = process_page(
                page_num, client, model_id, instructions, example_parts, base_dir
            )

            error_msg = data.get("error", "")

            # Determine if we should retry
            should_retry = False
            retry_type = None

            if not success:
                if "RESOURCE_EXHAUSTED" in error_msg or "429" in error_msg:
                    should_retry = True
                    retry_type = "rate_limit"
                    max_retries = max_retries_rate_limit
                    retry_wait = min(base_delay * (2 ** retry_count), max_delay)
                elif "Empty response" in error_msg or "NO_CANDIDATES" in error_msg:
                    should_retry = True
                    retry_type = "empty_response"
                    max_retries = max_retries_other
                    retry_wait = 2
                elif "not subscriptable" in error_msg or "NoneType" in error_msg:
                    should_retry = True
                    retry_type = "api_error"
                    max_retries = max_retries_other
                    retry_wait = 2
                elif "PDF not found" in error_msg:
                    # Don't retry for missing files
                    should_retry = False
                else:
                    # Unknown error - try once more
                    should_retry = retry_count == 0
                    retry_type = "unknown"
                    max_retries = 1
                    retry_wait = 2

            if should_retry and retry_count < max_retries:
                retry_count += 1
                print(f"⚠ {retry_type} error (attempt {retry_count}/{max_retries}): {error_msg}")
                print(f"Retrying in {retry_wait} seconds...")
                time.sleep(retry_wait)
                continue
            else:
                # Either success or max retries reached
                final_data = data
                break

        # Save results
        if success:
            # Save to JSONL
            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            # Save individual file
            with open(individual_file, 'w', encoding='utf-8') as f:
                json.dump(final_data, f, ensure_ascii=False, indent=2)

            retry_msg = f" (after {retry_count} retries)" if retry_count > 0 else ""
            print(f"✓ Page {page_num} saved{retry_msg}")
            results_summary.append({
                "page": page_num,
                "status": "success",
                "retries": retry_count
            })
            wait_time = base_delay

        else:
            print(f"✗ Page {page_num} failed: {final_data.get('error', 'Unknown error')}")

            # Save failed attempt to JSONL for debugging
            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            results_summary.append({
                "page": page_num,
                "status": "failed",
                "error": final_data.get('error'),
                "finish_reason": final_data.get('finish_reason'),
                "retries": retry_count
            })

            # Exponential backoff for consecutive failures
            wait_time = min(base_delay * 2, max_delay)

        # Rate limiting between pages
        if page_num < end_page:
            print(f"Waiting {wait_time} seconds before next page...")
            time.sleep(wait_time)

    # Save summary
    summary_file = output_dir / 'processing_summary.json'
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump({
            "total_pages": len(results_summary),
            "successful": sum(1 for r in results_summary if r["status"] == "success"),
            "failed": sum(1 for r in results_summary if r["status"] == "failed"),
            "skipped": sum(1 for r in results_summary if r["status"] == "skipped"),
            "results": results_summary
        }, f, indent=2)

    print(f"\n{'='*50}")
    print(f"Processing complete!")
    print(f"Successful: {sum(1 for r in results_summary if r['status'] == 'success')}")
    print(f"Failed: {sum(1 for r in results_summary if r['status'] == 'failed')}")
    print(f"Skipped: {sum(1 for r in results_summary if r['status'] == 'skipped')}")
    print(f"Results saved to: {output_dir}")
    print(f"{'='*50}")

    return results_summary


In [76]:
base_directory = "montagu_1816_fr-en_v1"
# kocasekbanbasi vasif_1794-1805 vasif_1782-1787 vasif_1774-1779 vasif_1752-1774 enveri_1774-1783 enveri_1787-1792 enveri_1768-1774 nuri_1794-1799 mustafarasih_sefaretname_1793
# ahmedcavid_1790-1791 ahmedcavid_muntehabat ahmedcevdet_mukaddime ahmedcevdet_1774-1783 ahmedcevdet_1783-1786 ahmedresmi_1763-1778 cabi_1788-1814 caniklialipasa_risale
# defterdarmehmed_layiha_cagman defterdarmehmed_layiha_ozdemir ebubekirratib_sefaretname fuadefendi_layiha cesmizade_1766-1768 ebubekirratib_mektup said_1806-1809 vakayi_eflak_1717
# sirkatibiahmed_1791-1807 ruzname_1769-1774 necati_rusyasefaretnamesi silahdar_1654-1695 silahdar_1695-1721 celebizade_1722-1729 rasid_1703-1722 rasid_1660-1703 mahmudraif_nizami_cedid
# kececizadeizzetmolla_layiha islahat_layihalari_cagman hakim_1752-1766 subhi_1730-1744 layiha_dagli_isyani layiha_spain ibrahimmuteferrika_usuluhikem vasif_tesliyetname
# abdurrahimmuhibefendi_kucuksefaretname ahmeddurriefendi_iransefaretnamesi behicefendi_layiha seyyidmehmedemin_fransasefaretnamesi seyyidmehmedemin_mukalemename dohsson_layiha
# abdurrahimmuhibefendi_buyuksefaretname asim_1804-1807 asim_1807-1809 ahmedresmi_hamiletulkubera edib_1788-1792 sanizade_1808-1815 sanizade_1816-1821 bahirefendi_1821-1826
# esad_ussizafer esad_1821-1826 sehdiosmanefendi_rusyasefaretnamesi giray_gulbuni_hanan giridlihaciahmed_1760 kirimlimustafarahmi_iransefaretnamesi kocaragibmehmed_fethiyyeibelgrad
# kocaragibmehmed_munseat kocasinanpasa_telhisler mustafakesbi_ibretnumayidevlet risalei_terceme_1785 nislimehmedaga_rusyasefaretnamesi mustafahattiefendi_viyanasefaretnamesi
# nizami_cedid_kanunlari ruzname_1757-1763 saidbhalil_seferirusya seyfullahaga_viyanasefaretnamesi seyyidmustafa_risale seyyidmehmedrefi_iransefaretnamesi tavukcureismustafa_viyanasefaretnamesi
# tarihi_varadin_1694 yaylaimami_risale ubeydullahkusmani_risale ubeydullahkusmani_ve_ebubekirefendi_risale yasincizadeabdulvehhab_iransefaretnamesi yirmisekizcelebimehmed_fransasefaretnamesi
# destarisalih_vakaiibretnuma abdulgaffarkirimi_1747 taylesanizadeabdullah_1785-1789 ebubekirratib_layiha risalei_garibe_1720 kefeliibrahim_tevarihitatar
# islahat_layihalari_karal ebubekirratib_layiha_tables abdi_1729-1731 semdanizadesuleyman_1730-1755 semdanizadesuleyman_1755-1769 semdanizadesuleyman_1769-1774
# semdanizadesuleyman_1774-1777

In [73]:
# Example usage:

# Process the first 5 pages
results = batch_process_pages(
    start_page=9,
    end_page=15,
    client=client,
    model_id=MODEL_ID,
    instructions=instructions,
    example_parts=example_parts,
    base_dir=base_directory
)

Processing page 9...
✓ Page 9 saved
Waiting 5 seconds before next page...
Processing page 10...
✓ Page 10 saved
Waiting 5 seconds before next page...
Processing page 11...
✓ Page 11 saved
Waiting 5 seconds before next page...
Processing page 12...
✓ Page 12 saved
Waiting 5 seconds before next page...
Processing page 13...
✓ Page 13 saved
Waiting 5 seconds before next page...
Processing page 14...
✓ Page 14 saved
Waiting 5 seconds before next page...
Processing page 15...
✓ Page 15 saved

Processing complete!
Successful: 7
Failed: 0
Skipped: 0
Results saved to: montagu_fr_1764/gemini_output


In [ ]:

def get_remaining_pages(base_dir):
    """
    Identify which pages have already been processed and which remain.

    Args:
        base_dir: Base directory containing transliteration_pages and gemini_output folders

    Returns:
        tuple: (processed_pages, remaining_pages, all_page_numbers)
    """
    base_path = Path(base_dir)

    # Check which pages have already been processed
    output_dir = base_path / 'gemini_output'
    if output_dir.exists():
        processed_pages = {int(p.stem.split('_')[1]) for p in output_dir.glob('page_*.json')}
    else:
        processed_pages = set()

    print(f"Already processed: {sorted(processed_pages) if processed_pages else 'None'}")

    # Get all available page PDFs
    transliteration_dir = base_path / 'transliteration_pages'
    if transliteration_dir.exists():
        page_files = os.listdir(transliteration_dir)
        all_page_numbers = [int(re.search(r'page_(\d+)\.pdf', f).group(1))
                            for f in page_files
                            if f.startswith("page_") and f.endswith(".pdf")]
    else:
        print(f"Warning: {transliteration_dir} does not exist!")
        all_page_numbers = []

    # Calculate remaining pages
    if all_page_numbers:
        remaining_pages = [p for p in range(min(all_page_numbers), max(all_page_numbers) + 1)
                           if p not in processed_pages]
        print(f"Total pages available: {len(all_page_numbers)}")
        print(f"Remaining to process: {len(remaining_pages)} pages")
        if remaining_pages:
            print(f"Page range: {min(remaining_pages)} to {max(remaining_pages)}")
    else:
        remaining_pages = []
        print("No pages found to process")

    return processed_pages, remaining_pages, all_page_numbers

In [79]:
#base_directory = ""

processed, remaining, all_pages = get_remaining_pages(base_directory)


Already processed: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 194, 195, 196, 198, 199, 200, 202, 206, 207, 209, 210, 211, 214, 218, 220, 226]
Total pages available: 404
Remaining to process: 19

In [78]:
# Process only the remaining pages
if remaining:
  print(f"\nStarting batch processing from page {remaining[0]}...")
  results = batch_process_pages(
    start_page=min(remaining),
    end_page=max(remaining),
    client=client,
    model_id=MODEL_ID,
    instructions=instructions,
    example_parts=example_parts,
    base_dir=base_directory,
    skip_existing=True
  )
else:
    print("All pages already processed!")


Starting batch processing from page 2...
Processing page 2...
✓ Page 2 saved
Waiting 5 seconds before next page...
Processing page 3...
✓ Page 3 saved
Waiting 5 seconds before next page...
Processing page 4...
✓ Page 4 saved
Waiting 5 seconds before next page...
Processing page 5...
✓ Page 5 saved
Waiting 5 seconds before next page...
Processing page 6...
✓ Page 6 saved
Waiting 5 seconds before next page...
Processing page 7...
✓ Page 7 saved
Waiting 5 seconds before next page...
Processing page 8...
✓ Page 8 saved
Waiting 5 seconds before next page...
Processing page 9...
✓ Page 9 saved
Waiting 5 seconds before next page...
Processing page 10...
✓ Page 10 saved
Waiting 5 seconds before next page...
Processing page 11...
✓ Page 11 saved
Waiting 5 seconds before next page...
Processing page 12...
✓ Page 12 saved
Waiting 5 seconds before next page...
Processing page 13...
✓ Page 13 saved
Waiting 5 seconds before next page...
Processing page 14...
✓ Page 14 saved
Waiting 5 seconds before

KeyboardInterrupt: 

We will check for completeness and then re-run the files that do no have an output

In [ ]:
file_name = base_directory

In [ ]:
# Path setup
pdf_path = Path(f"/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/{file_name}.pdf")
output_dir = Path(f"/content/drive/MyDrive/merve-february2026/{file_name}/gemini_output")

# Check if PDF exists
if not pdf_path.exists():
    print(f"PDF not found: {pdf_path}")
else:
    # Get number of pages in PDF
    reader = PdfReader(pdf_path)
    pdf_page_count = len(reader.pages)
    print(f"PDF has {pdf_page_count} pages")

    # Count JSON files in output directory
    if not output_dir.exists():
        print(f"Output directory not found: {output_dir}")
        json_file_count = 0
    else:
        json_files = list(output_dir.glob("page_*.json"))
        json_file_count = len(json_files)
        print(f"Found {json_file_count} JSON files in output directory")

    # Compare
    if pdf_page_count == json_file_count:
        print(f"All pages processed! ({pdf_page_count} pages)")
    else:
        missing = pdf_page_count - json_file_count
        print(f"Missing {missing} pages (PDF: {pdf_page_count}, JSON files: {json_file_count})")

PDF has 682 pages
Found 677 JSON files in output directory
Missing 5 pages (PDF: 682, JSON files: 677)


In [ ]:
# Check for files with null raw_response
if not output_dir.exists():
    print(f"Output directory not found: {output_dir}")
else:
    json_files = sorted(output_dir.glob("page_*.json"))

    if not json_files:
        print("No JSON files found")
    else:
        null_responses = []

        for json_file in json_files:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

                # Check if raw_response is null or missing
                if data.get('raw_response') is None:
                    page_num = data.get('page', 'unknown')
                    null_responses.append({
                        'file': json_file.name,
                        'page': page_num
                    })

        # Report results
        total_files = len(json_files)
        if not null_responses:
            print(f"All {total_files} files have text output")
        else:
            print(f"Found {len(null_responses)} files with null raw_response out of {total_files} total:")
            for item in null_responses:
                print(f"  - {item['file']}")

In [ ]:
# def batch_process_specific_pages(page_list, client, model_id, instructions, example_parts, base_dir, output_suffix="second_pass"):
#     """
#     Process specific pages (useful for reprocessing failed pages).

#     Args:
#         page_list: List of specific page numbers to process
#         client: Gemini client
#         model_id: Model identifier
#         instructions: Processing instructions
#         example_parts: Few-shot examples
#         base_dir: Base directory containing the transliteration_pages folder
#         output_suffix: Suffix for output directory (default: "second_pass")
#                       Use None to overwrite original files
#     """
#     # Determine output directory
#     if output_suffix:
#         output_dir = Path(base_dir) / f'gemini_output_{output_suffix}'
#     else:
#         output_dir = Path(base_dir) / 'gemini_output'

#     output_dir.mkdir(parents=True, exist_ok=True)

#     # JSONL file for continuous append
#     jsonl_file = output_dir / 'all_responses.jsonl'

#     # Track progress
#     results_summary = []

#     # Retry parameters
#     base_delay = 5
#     max_delay = 120
#     max_retries_429 = 5  # For rate limits
#     max_retries_empty = 3  # For empty/blocked responses
#     consecutive_failures = 0

#     for i, page_num in enumerate(page_list, 1):
#         print(f"[{i}/{len(page_list)}] Processing page {page_num}...")

#         # Retry loop
#         retry_count = 0
#         success = False
#         final_data = None

#         while retry_count <= max_retries_429:
#             success, data = process_page(
#                 page_num,
#                 client,
#                 model_id,
#                 instructions,
#                 example_parts,
#                 base_dir
#             )

#             error_msg = data.get("error", "")

#             # Handle RESOURCE_EXHAUSTED (429) errors
#             if not success and "RESOURCE_EXHAUSTED" in error_msg:
#                 retry_count += 1

#                 if retry_count <= max_retries_429:
#                     retry_wait = min(base_delay * (2 ** retry_count), max_delay)
#                     print(f"⚠ 429 Error (attempt {retry_count}/{max_retries_429})")
#                     print(f"Waiting {retry_wait} seconds...")
#                     time.sleep(retry_wait)
#                     continue
#                 else:
#                     print(f"✗ Page {page_num} failed after {max_retries_429} rate limit retries")
#                     final_data = data
#                     break

#             # Handle empty responses (blocked/safety issues)
#             elif not success and "Empty response" in error_msg:
#                 retry_count += 1

#             # Handle subscript/API errors - ADD THIS
#             elif not success and ("not subscriptable" in error_msg or "NoneType" in error_msg):
#                 retry_count += 1

#                 if retry_count <= max_retries_empty:
#                     print(f"⚠ Empty response (attempt {retry_count}/{max_retries_empty}): {data.get('finish_reason', 'unknown')}")
#                     print(f"Retrying in 2 seconds...")
#                     time.sleep(2)
#                     continue
#                 else:
#                     print(f"✗ Page {page_num} blocked after {max_retries_empty} attempts")
#                     final_data = data
#                     break

#             # Success or non-retryable error
#             final_data = data
#             break

#         # Save results
#         if success:
#             # 1. Append to JSONL immediately
#             with open(jsonl_file, 'a', encoding='utf-8') as f:
#                 f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

#             # 2. Save individual JSON file
#             individual_file = output_dir / f'page_{page_num}.json'
#             with open(individual_file, 'w', encoding='utf-8') as f:
#                 json.dump(final_data, f, ensure_ascii=False, indent=2)

#             retry_msg = f" (after {retry_count} retries)" if retry_count > 0 else ""
#             print(f"✓ Page {page_num} saved{retry_msg}")
#             results_summary.append({
#                 "page": page_num,
#                 "status": "success",
#                 "retries": retry_count
#             })

#             consecutive_failures = 0
#             wait_time = base_delay

#         else:
#             print(f"✗ Page {page_num} failed: {final_data.get('error', 'Unknown error')}")

#             with open(jsonl_file, 'a', encoding='utf-8') as f:
#                 f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

#             results_summary.append({
#                 "page": page_num,
#                 "status": "failed",
#                 "error": final_data.get('error'),
#                 "finish_reason": final_data.get('finish_reason')
#             })

#             # Only increase backoff for non-PDF-not-found errors
#             if "PDF not found" not in final_data.get('error', ''):
#                 consecutive_failures += 1
#                 wait_time = min(base_delay * (2 ** consecutive_failures), max_delay)
#             else:
#                 wait_time = base_delay

#         # Rate limiting between pages
#         if i < len(page_list):  # Don't wait after the last page
#             print(f"Waiting {wait_time} seconds before next page...")
#             time.sleep(wait_time)

#     # Save summary
#     summary_file = output_dir / 'processing_summary.json'
#     with open(summary_file, 'w', encoding='utf-8') as f:
#         json.dump({
#             "total_pages": len(results_summary),
#             "successful": sum(1 for r in results_summary if r["status"] == "success"),
#             "failed": sum(1 for r in results_summary if r["status"] == "failed"),
#             "pages_processed": page_list,
#             "results": results_summary
#         }, f, indent=2)

#     print(f"\n{'='*50}")
#     print(f"Processing complete!")
#     print(f"Successful: {sum(1 for r in results_summary if r['status'] == 'success')}")
#     print(f"Failed: {sum(1 for r in results_summary if r['status'] == 'failed')}")
#     print(f"Results saved to: {output_dir}")
#     print(f"{'='*50}")

#     return results_summary

In [ ]:
# second version

def batch_process_specific_pages(page_list, client, model_id, instructions, example_parts, base_dir, output_suffix="second_pass"):
    """
    Process specific pages (e.g., failed pages from a previous run).

    Args:
        page_list: List of page numbers to process
        output_suffix: If provided, creates separate output folder (e.g., "second_pass")
    """
    if output_suffix:
        output_dir = Path(base_dir) / f'gemini_output_{output_suffix}'
    else:
        output_dir = Path(base_dir) / 'gemini_output'

    output_dir.mkdir(parents=True, exist_ok=True)

    jsonl_file = output_dir / 'all_responses.jsonl'
    results_summary = []

    # Retry parameters
    base_delay = 10
    max_delay = 120
    max_retries_rate_limit = 5
    max_retries_other = 3

    for idx, page_num in enumerate(page_list, 1):
        print(f"[{idx}/{len(page_list)}] Processing page {page_num}...")

        # Retry loop
        retry_count = 0
        success = False
        final_data = None

        while retry_count < max_retries_rate_limit:
            success, data = process_page(
                page_num, client, model_id, instructions, example_parts, base_dir
            )

            error_msg = data.get("error", "")

            # Determine if we should retry
            should_retry = False
            retry_type = None

            if not success:
                if "RESOURCE_EXHAUSTED" in error_msg or "429" in error_msg:
                    should_retry = True
                    retry_type = "rate_limit"
                    max_retries = max_retries_rate_limit
                    retry_wait = min(base_delay * (2 ** retry_count), max_delay)
                elif "Empty response" in error_msg or "NO_CANDIDATES" in error_msg:
                    should_retry = True
                    retry_type = "empty_response"
                    max_retries = max_retries_other
                    retry_wait = 3
                elif "not subscriptable" in error_msg or "NoneType" in error_msg:
                    should_retry = True
                    retry_type = "api_error"
                    max_retries = max_retries_other
                    retry_wait = 3
                else:
                    should_retry = retry_count == 0
                    retry_type = "unknown"
                    max_retries = 1
                    retry_wait = 2

            if should_retry and retry_count < max_retries:
                retry_count += 1
                print(f"⚠ {retry_type} (attempt {retry_count}/{max_retries}): {error_msg}")
                print(f"Retrying in {retry_wait} seconds...")
                time.sleep(retry_wait)
                continue
            else:
                final_data = data
                break

        # Save results
        individual_file = output_dir / f'page_{page_num}.json'

        if success:
            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            with open(individual_file, 'w', encoding='utf-8') as f:
                json.dump(final_data, f, ensure_ascii=False, indent=2)

            retry_msg = f" (after {retry_count} retries)" if retry_count > 0 else ""
            print(f"✓ Page {page_num} saved{retry_msg}")
            results_summary.append({
                "page": page_num,
                "status": "success",
                "retries": retry_count
            })

        else:
            print(f"✗ Page {page_num} failed: {final_data.get('error', 'Unknown error')}")

            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            results_summary.append({
                "page": page_num,
                "status": "failed",
                "error": final_data.get('error'),
                "finish_reason": final_data.get('finish_reason'),
                "retries": retry_count
            })

        # Wait before next page
        if idx < len(page_list):
            print(f"Waiting {base_delay} seconds before next page...")
            time.sleep(base_delay)

    # Save summary
    summary_file = output_dir / 'processing_summary.json'
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump({
            "total_pages": len(results_summary),
            "successful": sum(1 for r in results_summary if r["status"] == "success"),
            "failed": sum(1 for r in results_summary if r["status"] == "failed"),
            "results": results_summary
        }, f, indent=2)

    print(f"\n{'='*50}")
    print(f"Processing complete!")
    print(f"Successful: {sum(1 for r in results_summary if r['status'] == 'success')}")
    print(f"Failed: {sum(1 for r in results_summary if r['status'] == 'failed')}")
    print(f"Results saved to: {output_dir}")
    print(f"{'='*50}")

    return results_summary


In [ ]:
from pathlib import Path
from pypdf import PdfReader

# Path setup
pdf_path = Path(f"/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/{file_name}.pdf")
transliteration_pages_dir = Path(f"/content/drive/MyDrive/merve-february2026/{file_name}/transliteration_pages")
output_dir = Path(f"/content/drive/MyDrive/merve-february2026/{file_name}/gemini_output")

# Check if PDF exists
if not pdf_path.exists():
    print(f"PDF not found: {pdf_path}")
    missing_pages = []
else:
    # Get number of pages in PDF
    reader = PdfReader(pdf_path)
    pdf_page_count = len(reader.pages)
    print(f"PDF has {pdf_page_count} pages")

    # Get the actual page range from transliteration_pages folder
    if not transliteration_pages_dir.exists():
        print(f"Transliteration pages directory not found: {transliteration_pages_dir}")
        missing_pages = []
    else:
        # Get all PDF files in transliteration_pages
        page_pdfs = list(transliteration_pages_dir.glob("page_*.pdf"))

        if not page_pdfs:
            print(f"No page PDFs found in {transliteration_pages_dir}")
            missing_pages = []
        else:
            # Extract page numbers from PDF filenames
            pdf_page_numbers = set()
            for page_pdf in page_pdfs:
                try:
                    page_num = int(page_pdf.stem.split('_')[1])
                    pdf_page_numbers.add(page_num)
                except (IndexError, ValueError):
                    print(f"Warning: Could not parse page number from {page_pdf.name}")

            if not pdf_page_numbers:
                print("Could not extract any page numbers from PDF files")
                missing_pages = []
            else:
                start_page = min(pdf_page_numbers)
                end_page = max(pdf_page_numbers)
                print(f"Page range from transliteration PDFs: {start_page} to {end_page} ({len(pdf_page_numbers)} pages)")

                # Get processed JSON files
                if not output_dir.exists():
                    print(f"Output directory not found: {output_dir}")
                    processed_pages = set()
                else:
                    json_files = list(output_dir.glob("page_*.json"))
                    json_file_count = len(json_files)
                    print(f"Found {json_file_count} JSON files in output directory")

                    # Extract page numbers from JSON filenames
                    processed_pages = set()
                    for json_file in json_files:
                        try:
                            page_num = int(json_file.stem.split('_')[1])
                            processed_pages.add(page_num)
                        except (IndexError, ValueError):
                            print(f"Warning: Could not parse page number from {json_file.name}")

                # Find missing pages (pages that exist in PDF but not in JSON)
                missing_pages = sorted(list(pdf_page_numbers - processed_pages))

                # Compare
                if len(pdf_page_numbers) == len(processed_pages) and not missing_pages:
                    print(f"✓ All pages processed! ({len(pdf_page_numbers)} pages, from {start_page} to {end_page})")
                else:
                    print(f"✗ Missing {len(missing_pages)} pages")
                    print(f"   Expected: {len(pdf_page_numbers)} pages ({start_page}-{end_page})")
                    print(f"   Processed: {len(processed_pages)} JSON files")
                    print(f"   Missing page numbers: {missing_pages}")

# Now you can use the missing_pages list
print(f"\nMissing pages list: {missing_pages}")

In [ ]:
missing_pages

In [ ]:


# Step 1: Find pages with no response
pages_to_reprocess = [366]


# Step 2: Reprocess to a separate folder (recommended)
if pages_to_reprocess:
    print(f"\nReprocessing {len(pages_to_reprocess)} pages to gemini_output_second_pass...")
    results = batch_process_specific_pages(
        page_list=pages_to_reprocess,
        client=client,
        model_id=MODEL_ID,
        instructions=instructions,
        example_parts=example_parts,
        base_dir=base_directory,
        output_suffix="second_pass"  # Creates gemini_output_second_pass
    )
else:
    print("All pages have valid responses!")

# Optional: If you want to overwrite instead, use:
# output_suffix=None

Reprocessing 2 pages to gemini_output_second_pass...
[1/2] Processing page 75...
✗ Page 75 failed: Empty response (finish_reason: NO_CANDIDATES)
Waiting 10 seconds before next page...
[2/2] Processing page 366...
⚠ 429 Error (attempt 1/5)
Waiting 10 seconds...
⚠ 429 Error (attempt 2/5)
Waiting 20 seconds...
✗ Page 366 failed: Empty response (finish_reason: NO_CANDIDATES)

==================================================
Processing complete!
Successful: 0
Failed: 2
Results saved to: ahmedcevdet_1774-1783/gemini_output_second_pass

Reprocessing 1 pages to gemini_output_second_pass...
[1/1] Processing page 185...
✗ Page 185 failed: 'NoneType' object is not subscriptable


In [ ]:
base_directory = 'ahmedcevdet_mukaddime'

# Step 1: Find pages with no response
pages_to_reprocess = [375]


# Step 2: Reprocess to a separate folder (recommended)
if pages_to_reprocess:
    print(f"\nReprocessing {len(pages_to_reprocess)} pages to gemini_output_second_pass...")
    results = batch_process_specific_pages(
        page_list=pages_to_reprocess,
        client=client,
        model_id=MODEL_ID,
        instructions=instructions,
        example_parts=example_parts,
        base_dir=base_directory,
        output_suffix="second_pass"  # Creates gemini_output_second_pass
    )
else:
    print("All pages have valid responses!")

# Optional: If you want to overwrite instead, use:
# output_suffix=None

for whatever reason ahmedcevdet_mukaddime page 375 is not working. i keep getting nonetype object error